In [27]:
import os
import numpy as np
from scipy.optimize import minimize
from topological_insulator import Problem

In [28]:
structure_path = "../../../../../topological_insulator/data/structures/"
structure_name = "honeycomb.json"

In [ ]:
class InteractionOptimizer:
    def __init__(self, Delta_SOC=-5, t=-1, delta=0.832, U=1.5, groundstate_band=15):
        # Load initial wavefunction
        Psi_r = np.loadtxt("non_interacting_coefficients.txt", dtype=complex)
        self.Psi_r_init = Psi_r
        self.occupation_init = np.abs(Psi_r)**2
        self.structure_path = structure_path
        self.structure_name = structure_name
        self.t = t
        self.Delta_SOC = Delta_SOC
        self.delta = delta
        self.U = U
        self.groundstate_band = groundstate_band
        os.mkdir("coefficients")

    def _fitness(self, occupation_old):
        print("")
        problem = Problem(structure_path=structure_path, structure_name=structure_name)
        self._set_eigenvalues(problem, occupation_old)
        problem.setup(
            N_r = 10,
            N_k = 400,
            location = "bulk",
            BZ = "reduced"
        )
        problem.run(
            H_type="reciprocal"
        )
        Psi_r = self.get_eigenvector(problem)
        occupation_new = np.abs(Psi_r)**2
        return occupation_new

    def objective(self, occupation_old):
        occupation_new = self._fitness(occupation_old)
        rmse = np.sqrt(np.mean(np.abs(occupation_new - occupation_old)**2))
        np.savetxt(f"coefficients/interacting_coefficients_{rmse}.txt", occupation_new)
        return rmse

    def get_bounds(self):
        n = len(self.Psi_r_init)
        return [(0, 1) for _ in range(n)]

    def get_constraint(self):
        return {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}

    def run_optimization(self):
        x0 = self.occupation_init
        bounds = self.get_bounds()
        constraints = [self.get_constraint()]

        result = minimize(
            self.objective,
            x0,
            method='trust-constr',
            bounds=bounds,
            constraints=constraints,
            options={
            'xtol': 1e-6,
            'gtol': 1e-6,
            # 'ftol': 1e-6, 
            'disp': True, 
            'maxiter': 500
            }
        )
        return result
    
    def _set_eigenvalues(self, problem:Problem, occupation_old):
        sublattice_labels = ["A", "B", "C", "D", "E", "F"]
        cell = problem.cell_parser
        g = cell.geometry
        n_subs = len(g.delta_vectors.value)
        subs = sublattice_labels[:n_subs]
        for i, label_i in enumerate(subs):
            parser = getattr(problem.cell_parser.eigenvalues, label_i).value
            # Diagonal Values
            parser["chadi_soc"][label_i]["Delta_pp"] = self.Delta_SOC
            parser["interaction"][label_i]["U_p"] = self.U
            parser["interaction"][label_i]["n_px_up"] = occupation_old[(i+1)*2]
            parser["interaction"][label_i]["n_px_down"] = occupation_old[(i+1)*3]
            parser["interaction"][label_i]["n_py_up"] = occupation_old[(i+1)*4]
            parser["interaction"][label_i]["n_py_down"] = occupation_old[(i+1)*5]
            parser["interaction"][label_i]["n_pz_up"] = occupation_old[(i+1)*6]
            parser["interaction"][label_i]["n_pz_down"] = occupation_old[(i+1)*7]
            # Off-Diagonal Values
            for label_j in subs:
                # Hoppings
                try:
                    parser["nn_hopping"][label_j]["t_ss_sigma"] = 0.01 * self.t
                    # parser["nn_hopping"][label_j]["t_sp_sigma"] = 1
                    parser["nn_hopping"][label_j]["t_pp_sigma"] = self.t - self.delta
                    parser["nn_hopping"][label_j]["t_pp_pi"] = self.t + self.delta
                except:
                    pass

    def get_eigenvector(self, problem:Problem):
        n = self.groundstate_band
        g = problem.geometry
        tb_bulk = problem.hamiltonian["bulk"]["tight_binding"]
        Psi_k_0 = tb_bulk.band_structure_data["eigenvector_dict"][n]
        data = tb_bulk.sublattice_data_dict
        idx_A, idx_B = data["A"]["idx"], data["B"]["idx"]
        r_A, r_B = g.sites[idx_A], g.sites[idx_B]
        N_projections = len(tb_bulk.coupled_states)
        slice_A = slice(0 * N_projections, (0 + 1) * N_projections)
        slice_B = slice(1 * N_projections, (1 + 1) * N_projections)
        Psi_r_0 = np.zeros(shape=Psi_k_0[0].shape, dtype=complex)
        for k_idx, c_k_i in enumerate(Psi_k_0):
            k = np.array(tb_bulk.band_structure_data["k_bulk"][k_idx])
            # sublattice A
            phase_A = np.exp(-1j * np.dot(k, r_A))
            Psi_r_0[slice_A] += phase_A * c_k_i[slice_A]
            # sublattice B
            phase_B = np.exp(-1j * np.dot(k, r_B))
            Psi_r_0[slice_B] += phase_B * c_k_i[slice_B]
        N =  len(Psi_k_0)
        Psi_r_0 /= np.sqrt(N)
        Psi_r_0 /= np.sqrt(np.sum(np.abs(Psi_r_0)**2)) # enforce normalization
        I = np.identity(2)
        M = tb_bulk.A.conj().T @ tb_bulk.C.conj().T
        S = np.kron(I, M)
        Psi_r = S @  Psi_r_0
        return Psi_r


In [30]:
optimizer = InteractionOptimizer()
res = optimizer.run_optimization()

print("Optimized occupation:", res.x)
print("Final RMSE:", res.fun)
print("Constraint sum:", np.sum(res.x))



Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bul

/home/jlgpke/miniconda3/lib/python3.12/site-packages/scipy/optimize/_differentiable_functions.py:551: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(delta_x, delta_g)



Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!

Building Geometry...
Geometry - Done.
Building 'Bul

KeyboardInterrupt: 